## Evaluate the run time, Mainboard for the baseline example of the BASEforHANK package.

# Header: set up paths, pre-process user inputs, load module

In [1]:
6-3

3

In [8]:
run(`git clone  https://github.com/lampe142/BASEforHANK_Cloud_CPU_GPU_TPU.git`)

Cloning into 'BASEforHANK_Cloud_CPU_GPU_TPU'...


Process(`git clone https://github.com/lampe142/BASEforHANK_Cloud_CPU_GPU_TPU.git`, ProcessExited(0))

In [10]:
pwd()

"/content/BASEforHANK_Cloud_CPU_GPU_TPU"

In [9]:
cd("BASEforHANK_Cloud_CPU_GPU_TPU")

In [12]:
include("setup.jl")

LoadError: LoadError: IOError: cd("/Users/max/colab/BASEforHANK_Cloud_CPU_GPU_TPU"): no such file or directory (ENOENT)
in expression starting at /content/BASEforHANK_Cloud_CPU_GPU_TPU/setup.jl:2

In [2]:
Base.current_project()
pwd()

"/content"

In [ ]:
Pkg.activate(".")

  Activating project at `~/colab/BASEforHANK_Cloud_CPU_GPU_TPU`


In [ ]:
using Pkg
Pkg.instantiate()
Pkg.precompile()

In [ ]:
using PrettyTables, Printf, BenchmarkTools, Dates;

root_dir = "/Users/max/colab/BASEforHANK_Cloud_CPU_GPU_TPU";
cd(root_dir);

# set up paths for the project
paths = Dict(
    "root" => root_dir,
    "src" => joinpath(root_dir, "src"),
    "bld" => joinpath(root_dir, "bld"),
    "src_example" => @__DIR__,
    "bld_example" => replace(@__DIR__, "examples" => "bld") * "_estim",
    "bld_runtime" => joinpath(root_dir, "bld/baseline_runtime"),
);

res_Runtime = Dict(
    "KERNEL" => Sys.KERNEL,
    "MACHINE" => Sys.MACHINE,
    "CPU_NAME" => Sys.CPU_NAME,
    "JULIA_VERSION" => VERSION,
    "nthreads" => Threads.nthreads(),
    "TOTAL_MEMORY" => Sys.total_memory() |> Base.format_bytes,
    "FREE_MEMORY" => Sys.free_memory() |> Base.format_bytes
);

Pkg.update(["DataFrames", "PrettyTables", "Crayons"])

# create bld directory for the current example
mkpath(paths["bld_example"]);


# pre-process user inputs for model setup
include(paths["src"] * "/Preprocessor/PreprocessInputs.jl");
include(paths["src"] * "/BASEforHANK.jl");

using .BASEforHANK;

# set BLAS threads to the number of Julia threads, prevents grabbing all
BASEforHANK.LinearAlgebra.BLAS.set_num_threads(Threads.nthreads());


    Updating registry at `~/.julia/registries/General`
┌ Info: The General registry is installed via git. Consider reinstalling it via
│ the newer faster direct from tarball format by running:
│   pkg> registry rm General; registry add General
│ 
└ @ Pkg.Registry /Applications/Julia-1.11.app/Contents/Resources/julia/share/julia/stdlib/v1.11/Pkg/src/Registry/Registry.jl:478
    Updating git-repo `https://github.com/JuliaRegistries/General.git`
  No Changes to `~/colab/BASEforHANK_Cloud_CPU_GPU_TPU/Project.toml`
  No Changes to `~/colab/BASEforHANK_Cloud_CPU_GPU_TPU/Manifest.toml`



Preprocessing Inputs...
Consistency check passed:
1. All variables have an error equation.
2. All variables with an error equation are in the variable list.
Preprocessing Inputs... Done.


# Initialize: set up model parameters, priors, and estimation settings

In [ ]:
paths["src"]

"/Users/max/colab/BASEforHANK_Cloud_CPU_GPU_TPU/src"

In [ ]:
include(paths["src"] * "/BASEforHANK.jl");

In [ ]:
# model parameters and priors
m_par = ModelParameters();
priors = collect(metaflatten(m_par, prior));
par_prior = mode.(priors);
m_par = BASEforHANK.Flatten.reconstruct(m_par, par_prior);
e_set = BASEforHANK.e_set;

# set some paths
@set! e_set.save_mode_file = paths["bld_example"] * "/HANK_mode.jld2";
@set! e_set.save_posterior_file = paths["bld_example"] * "/HANK_chain.jld2";
@set! e_set.mode_start_file = paths["src_example"] * "/Data/par_final_dict.txt";
@set! e_set.data_file = paths["src_example"] * "/Data/bbl_data_inequality.csv";

# fix seed for random number generation
BASEforHANK.Random.seed!(e_set.seed);

# Calculate Steady State and prepare linearization

In [ ]:
t = @timed call_find_steadystate(m_par);
res_Runtime["t_call_find_steadystate"] = t.time;
ss_full = t.value;

# sparse DCT representation
timing = @timed call_prepare_linearization(ss_full, m_par)
sr_full = timing.value
res_Runtime["t_call_prepare_linearization"] = timing.time


# save the steady state
jldsave(paths["bld_example"] * "/steadystate.jld2", true; sr_full);

# compute steady state moments
K = exp.(sr_full.XSS[sr_full.indexes.KSS]);
B = exp.(sr_full.XSS[sr_full.indexes.BSS]);
Bgov = exp.(sr_full.XSS[sr_full.indexes.BgovSS]);
Y = exp.(sr_full.XSS[sr_full.indexes.YSS]);
T10W = exp(sr_full.XSS[sr_full.indexes.TOP10WshareSS]);
G = exp.(sr_full.XSS[sr_full.indexes.GSS]);
fr_borr = BASEforHANK.eval_cdf(sr_full.distrSS, :b, sr_full.n_par, 0.0);

# Display steady state moments
@printf "\n"
pretty_table(
    [
        "Liquid to Illiquid Assets Ratio" B/K
        "Capital to Output Ratio" K / Y/4.0
        "Government Debt to Output Ratio" Bgov / Y/4.0
        "Government Spending to Output Ratio" G/Y
        "TOP 10 Wealth Share" T10W
        "Fraction of Borrower" fr_borr
    ];
    header = ["Variable", "Value"],
    title = "Steady State Moments",
    formatters = ft_printf("%.4f"),
)


Compute the steady state...
CompMarketsCapital function is defined, used for guesses.
Kmin: 21.021237, Kmax: 51.356145
Find capital stock, coarse income grid.
EGM: Iterations 664, Distance 9.91e-07, Time 0.53


┌ Warning: Non monotone resource list encountered: 
└ @ Main.BASEforHANK.SteadyState /Users/max/colab/BASEforHANK_Cloud_CPU_GPU_TPU/src/SubModules/SteadyState/EGM/EGM_policyupdate.jl:665


Sorting the ressource, consumption, liquid and capital lists. If this persists, adjust the grid! 
Sorting the ressource, consumption, liquid and capital lists. If this persists, adjust the grid! 
Sorting the ressource, consumption, liquid and capital lists. If this persists, adjust the grid! 
EGM: Iterations 1989, Distance 9.95e-07, Time 0.12
EGM: Iterations 1975, Distance 9.99e-07, Time 0.05


┌ Warning: Non monotone resource list encountered: 
└ @ Main.BASEforHANK.SteadyState /Users/max/colab/BASEforHANK_Cloud_CPU_GPU_TPU/src/SubModules/SteadyState/EGM/EGM_policyupdate.jl:665
┌ Warning: Non monotone resource list encountered: 
└ @ Main.BASEforHANK.SteadyState /Users/max/colab/BASEforHANK_Cloud_CPU_GPU_TPU/src/SubModules/SteadyState/EGM/EGM_policyupdate.jl:665


EGM: Iterations 561, Distance 9.97e-07, Time 0.02
EGM: Iterations 1021, Distance 9.98e-07, Time 0.02
EGM: Iterations 2755, Distance 9.97e-07, Time 0.07
EGM: Iterations 667, Distance 9.94e-07, Time 0.01
EGM: Iterations 1159, Distance 9.98e-07, Time 0.03
EGM: Iterations 847, Distance 9.99e-07, Time 0.02
EGM: Iterations 483, Distance 9.92e-07, Time 0.01
EGM: Iterations 2, Distance 9.62e-07, Time 0.00
EGM: Iterations 1, Distance 9.57e-07, Time 0.00
EGM: Iterations 1, Distance 9.51e-07, Time 0.00
EGM: Iterations 1, Distance 9.46e-07, Time 0.00
EGM: Iterations 1, Distance 9.43e-07, Time 0.00
EGM: Iterations 1, Distance 9.39e-07, Time 0.00
EGM: Iterations 1, Distance 9.36e-07, Time 0.00
EGM: Iterations 1, Distance 9.35e-07, Time 0.00
EGM: Iterations 1, Distance 9.35e-07, Time 0.00
EGM: Iterations 1, Distance 9.34e-07, Time 0.00
EGM: Iterations 1, Distance 9.34e-07, Time 0.00
EGM: Iterations 1, Distance 9.33e-07, Time 0.00
EGM: Iterations 1, Distance 9.33e-07, Time 0.00
EGM: Iterations 1, Dist

# Linearize the full model, find sparse state-space representation

In [ ]:
t = @timed  linearize_full_model(sr_full, m_par);
res_Runtime["t_linearize_full_model"] = t.time;
lr_full = t.value;

# save the linearization
jldsave(paths["bld_example"] * "/linearresults.jld2", true; lr_full);

# sparse state-space representation
t = @timed model_reduction(sr_full, lr_full, m_par);
res_Runtime["t_model_reduction"] = t.time;
sr_reduc =t.value;

lr_reduc = update_model(sr_reduc, lr_full, m_par);



# save the reduction
jldsave(paths["bld_example"] * "/reduction.jld2", true; sr_reduc, lr_reduc);

# model timing
@printf "One model solution takes: \n"
@set! sr_reduc.n_par.verbose = false;
@btime lr_reduc = update_model(sr_reduc, lr_full, m_par);
@set! sr_reduc.n_par.verbose = true;



Linearizing the full model...
Steady state is a solution!
Number of States and Controls: 1365
Max error on Fsys: 1.37e-07
Max error of distribution COP in Fsys: 1.13e-14
Max error of distribution b in Fsys: 1.51e-14
Max error of distribution k in Fsys: 1.81e-14
Max error of distribution h in Fsys: 4.86e-17
Max error of value function b in Fsys: 1.05e-13
Max error of value function k in Fsys: 4.92e-14
State Space Solution Done
Linearizing the full model... Done.

Model reduction (state-space representation)...
Reduction Step
Number of reduced model factors for DCTs for Wb & Wk: 30
Number of reduced model factors for copula DCTs: 27
Model reduction (state-space representation)... Done.
Updating linearization
One model solution takes: 
  150.676 ms (1590 allocations: 31.69 MiB)


# Estimation

In [ ]:
"Number of burn in iterations: " * string(e_set.burnin) * " and number of MCMC iterations: " * string(e_set.ndraws)

"Number of burn in iterations: 100 and number of MCMC iterations: 400"

In [ ]:
@set! e_set.burnin = 1000
@set! e_set.ndraws = 5000

EstimationSettings
  irf_matching: Bool false
  irf_matching_dict: Dict{Any, Any}
  shock_names: Array{Symbol}((9,))
  observed_vars_input: Array{Symbol}((11,))
  nobservables: Int64 11
  data_rename: Dict{Symbol, Symbol}
  me_treatment: Symbol unbounded
  me_std_cutoff: Float64 0.2
  meas_error_input: Array{Symbol}((2,))
  meas_error_distr: Array{Distributions.InverseGamma{Float64}}((2,))
  mode_start_file: String "/Users/max/colab/BASEforHANK_Cloud_CPU_GPU_TPU/examples/baseline/Data/par_final_dict.txt"
  data_file: String "/Users/max/colab/BASEforHANK_Cloud_CPU_GPU_TPU/examples/baseline/Data/bbl_data_inequality.csv"
  save_mode_file: String "/Users/max/colab/BASEforHANK_Cloud_CPU_GPU_TPU/bld/baseline_estim/HANK_mode.jld2"
  save_posterior_file: String "/Users/max/colab/BASEforHANK_Cloud_CPU_GPU_TPU/bld/baseline_estim/HANK_chain.jld2"
  estimate_model: Bool true
  max_iter_mode: Int64 3
  optimizer: Optim.NelderMead{Optim.AffineSimplexer, Optim.AdaptiveParameters}
  compute_hessian: B

In [ ]:

if e_set.estimate_model == true
    @printf "\n"
    @printf "Estimation...\n"

    # warning: estimation might take a long time!
    t = @timed er_mode, posterior_mode, smoother_mode, sr_mode, lr_mode, m_par_mode =
        find_mode(sr_reduc, lr_reduc, m_par, e_set)
    res_Runtime["t_find_mode"] = t.time;

    # Only relevant output for later plotting will be saved.
    # If you require all smoother output including the variance estimates
    # over time, items 4 and 5, comment out the next line.
    # This increases the hard disk storage significantly.
    smoother_mode = (0.0, 0.0, smoother_mode[3], 0.0, 0.0, smoother_mode[6], 0.0)

    # Stores mode finding results in file e_set.save_mode_file
    jldsave(
        e_set.save_mode_file,
        true;
        posterior_mode,
        smoother_mode,
        sr_mode,
        lr_mode,
        er_mode,
        m_par_mode,
        e_set,
    )
    # !! warning: the provided mode file does not contain smoothed covars (smoother_mode[4] and [5])!!
    # @load BASEforHANK.e_set.save_mode_file posterior_mode sr_mode lr_mode er_mode m_par_mode smoother_mode

    t = @belapsed begin
    sr_mc,
    lr_mc,
    er_mc,
    m_par_mc,
    draws_raw,
    posterior,
    accept_rate,
    par_final,
    hessian_sym,
    smoother_output = sample_posterior(sr_mode, lr_mode, er_mode, m_par_mode, e_set)
    end
    res_Runtime["t_sample_posterior"] = t;

    # Only relevant output for later plotting will be saved.
    # If you want all smoother output including the variance estimates
    # over time, items 4 and 5, comment out the next line.
    # This increases the hard disk storage significantly.
    smoother_output = (0.0, 0.0, smoother_output[3], 0.0, 0.0, smoother_output[6], 0.0)

    # Stores mcmc results in file e_set.save_posterior_file
    jldsave(
        e_set.save_posterior_file,
        true;
        sr_mc,
        lr_mc,
        er_mc,
        m_par_mc,
        draws_raw,
        posterior,
        accept_rate,
        par_final,
        hessian_sym,
        smoother_output,
        e_set,
    )
    # !! The following file is not provided !!
    #      @load BASEforHANK.e_set.save_posterior_file sr_mc lr_mc er_mc  m_par_mc draws_raw posterior accept_rate par_final hessian_sym smoother_output e_set

    @printf "Estimation... Done. \n"
end



Estimation...

Started mode finding. This might take a while...
Parameter try violates PRIOR.
Parameter try violates PRIOR.
Parameter try violates PRIOR.
Parameter try violates PRIOR.
Parameter try violates PRIOR.
Parameter try violates PRIOR.
Parameter try violates PRIOR.
Parameter try violates PRIOR.
Parameter try violates PRIOR.
Iter     Function value    √(Σ(yᵢ-ȳ)²)/n 
------   --------------    --------------
     0     3.540334e+03     7.867059e+15
 * time: 0.00020503997802734375
Parameter try violates PRIOR.
Parameter try violates PRIOR.
Parameter try violates PRIOR.
Parameter try violates PRIOR.
Parameter try violates PRIOR.
Parameter try violates PRIOR.
Parameter try violates PRIOR.
Parameter try violates PRIOR.
Parameter try violates PRIOR.
Parameter try violates PRIOR.
Parameter try violates PRIOR.
Parameter try violates PRIOR.
Updating model reduction after initial optimization.

Model reduction (state-space representation)...
Further model reduction switched off --> reve

UndefVarError: UndefVarError: `smoother_output` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

# Compute all IRFs, VDs, BCVDs, and historical decompositions

In [ ]:
@printf "\n"
@printf "Compute IRFs, VDs, and BCVDs...\n"

# Get indices of the shocks
exovars = [getfield(sr_mc.indexes_r, shock_names[i]) for i = 1:length(shock_names)];

# Get standard deviations of the shocks
stds_mc = [getfield(m_par_mc, Symbol("σ_", i)) for i in shock_names];
stds_mode = [getfield(m_par_mode, Symbol("σ_", i)) for i in shock_names];

# Compute IRFs
IRFs_mc, _, IRFs_order = compute_irfs(
    exovars,
    lr_mc.State2Control,
    lr_mc.LOMstate,
    sr_mc.XSS,
    sr_mc.indexes_r;
    init_val = stds_mc,
);
IRFs_mode, _, _, = compute_irfs(
    exovars,
    lr_mode.State2Control,
    lr_mode.LOMstate,
    sr_mode.XSS,
    sr_mc.indexes_r;
    init_val = stds_mode,
);

# Compute variance decomposition of IRFs
VDs_mc = compute_vardecomp(IRFs_mc);
VDs_mode = compute_vardecomp(IRFs_mode);

# Compute business cycle frequency variance decomposition
VDbcs_mc, UnconditionalVar_mc =
    compute_vardecomp_bcfreq(exovars, stds_mc, lr_mc.State2Control, lr_mc.LOMstate);
VDbcs_mode, UnconditionalVar_mode =
    compute_vardecomp_bcfreq(exovars, stds_mode, lr_mode.State2Control, lr_mode.LOMstate);

# Compute historical decompositions
ShockContr, ShockContr_order = compute_hist_decomp(
    exovars,
    lr_mc.State2Control,
    lr_mc.LOMstate,
    smoother_output,
    sr_mc.indexes_r,
);

# Graphical outputs

In [ ]:
@printf "\n"
@printf "Plotting...\n"

mkpath(paths["bld_example"] * "/IRFs");
plot_irfs(
    [
        (:Z, "TFP"),
        (:ZI, "Inv.-spec. tech."),
        (:μ, "Price markup"),
        (:μw, "Wage markup"),
        (:A, "Risk premium"),
        (:Rshock, "Mon. policy"),
        (:Gshock, "Structural deficit"),
        (:Tprogshock, "Tax progr."),
        (:Sshock, "Income risk"),
    ],
    [
        (:Ygrowth, "Output growth"),
        (:Cgrowth, "Consumption growth"),
        (:Igrowth, "Investment growth"),
        (:N, "Employment"),
        (:wgrowth, "Wage growth"),
        (:RB, "Nominal rate"),
        (:π, "Inflation"),
        (:σ, "Income risk"),
        (:Tprog, "Tax progressivity"),
        (:TOP10Wshare, "Top 10 wealth share"),
        (:TOP10Ishare, "Top 10 inc. share"),
    ],
    [(IRFs_mc, "Posterior mean"), (IRFs_mode, "Mode")],
    IRFs_order,
    sr_mc.indexes_r;
    show_fig = false,
    save_fig = true,
    path = paths["bld_example"] * "/IRFs",
    yscale = "standard",
    style_options = (lw = 2, color = [:blue, :red], linestyle = [:solid, :dash]),
)

mkpath(paths["bld_example"] * "/IRFs_cat");
plot_irfs_cat(
    Dict(
        ("Monetary", "mon") => [:Rshock, :A],
        ("Fiscal", "fis") => [:Gshock, :Tprogshock],
        ("Productivity", "pro") => [:Z, :ZI, :μ, :μw],
    ),
    [
        (:Ygrowth, "Output growth"),
        (:Cgrowth, "Consumption growth"),
        (:Igrowth, "Investment growth"),
        (:N, "Employment"),
        (:wgrowth, "Wage growth"),
        (:RB, "Nominal rate"),
        (:π, "Inflation"),
        (:σ, "Income risk"),
        (:Tprog, "Tax progressivity"),
        (:TOP10Wshare, "Top 10 wealth share"),
        (:TOP10Ishare, "Top 10 inc. share"),
    ],
    IRFs_mc,
    IRFs_order,
    sr_mc.indexes_r;
    show_fig = false,
    save_fig = true,
    path = paths["bld_example"] * "/IRFs_cat",
    yscale = "standard",
    style_options = (lw = 2, color = [:blue, :red], linestyle = [:solid, :dash]),
)

mkpath(paths["bld_example"] * "/VDs");
plot_vardecomp(
    [
        (:Ygrowth, "Output growth"),
        (:Cgrowth, "Consumption growth"),
        (:Igrowth, "Investment growth"),
        (:N, "Employment"),
        (:wgrowth, "Wage growth"),
        (:RB, "Nominal rate"),
        (:π, "Inflation"),
        (:σ, "Income risk"),
        (:Tprog, "Tax progressivity"),
        (:TOP10Wshare, "Top 10 wealth share"),
        (:TOP10Ishare, "Top 10 inc. share"),
    ],
    [(VDs_mc, "Posterior mean"), (VDs_mode, "Mode")],
    IRFs_order,
    sr_mc.indexes_r;
    show_fig = false,
    save_fig = true,
    path = paths["bld_example"] * "/VDs",
)

mkpath(paths["bld_example"] * "/VDs_cat");
plot_vardecomp(
    [
        (:Ygrowth, "Output growth"),
        (:Cgrowth, "Consumption growth"),
        (:Igrowth, "Investment growth"),
        (:N, "Employment"),
        (:wgrowth, "Wage growth"),
        (:RB, "Nominal rate"),
        (:π, "Inflation"),
        (:σ, "Income risk"),
        (:Tprog, "Tax progressivity"),
        (:TOP10Wshare, "Top 10 wealth share"),
        (:TOP10Ishare, "Top 10 inc. share"),
    ],
    [(VDs_mc, "Posterior mean"), (VDs_mode, "Mode")],
    IRFs_order,
    sr_mc.indexes_r;
    shock_categories = Dict(
        ("Monetary", "mon") => [:Rshock, :A],
        ("Fiscal", "fis") => [:Gshock, :Tprogshock],
        ("Productivity", "pro") => [:Z, :ZI, :μ, :μw],
    ),
    show_fig = false,
    save_fig = true,
    path = paths["bld_example"] * "/VDs_cat",
)

mkpath(paths["bld_example"] * "/VDbcs");
plot_vardecomp_bcfreq(
    [
        (:Ygrowth, "Output growth"),
        (:Cgrowth, "Consumption growth"),
        (:Igrowth, "Investment growth"),
        (:N, "Employment"),
        (:wgrowth, "Wage growth"),
        (:RB, "Nominal rate"),
        (:π, "Inflation"),
        (:σ, "Income risk"),
        (:Tprog, "Tax progressivity"),
        (:TOP10Wshare, "Top 10 wealth share"),
        (:TOP10Ishare, "Top 10 inc. share"),
    ],
    [(VDbcs_mc, "Posterior mean"), (VDbcs_mode, "Mode")],
    IRFs_order,
    sr_mc.indexes_r;
    show_fig = false,
    save_fig = true,
    path = paths["bld_example"] * "/VDbcs",
)

mkpath(paths["bld_example"] * "/VDbcs_cat");
plot_vardecomp_bcfreq(
    [
        (:Ygrowth, "Output growth"),
        (:Cgrowth, "Consumption growth"),
        (:Igrowth, "Investment growth"),
        (:N, "Employment"),
        (:wgrowth, "Wage growth"),
        (:RB, "Nominal rate"),
        (:π, "Inflation"),
        (:σ, "Income risk"),
        (:Tprog, "Tax progressivity"),
        (:TOP10Wshare, "Top 10 wealth share"),
        (:TOP10Ishare, "Top 10 inc. share"),
    ],
    [(VDbcs_mc, "Posterior mean"), (VDbcs_mode, "Mode")],
    IRFs_order,
    sr_mc.indexes_r;
    shock_categories = Dict(
        ("Monetary", "mon") => [:Rshock, :A],
        ("Fiscal", "fis") => [:Gshock, :Tprogshock],
        ("Productivity", "pro") => [:Z, :ZI, :μ, :μw],
    ),
    show_fig = false,
    save_fig = true,
    path = paths["bld_example"] * "/VDbcs_cat",
)

mkpath(paths["bld_example"] * "/HDs");
plot_hist_decomp(
    [
        (:Ygrowth, "Output growth"),
        (:Cgrowth, "Consumption growth"),
        (:Igrowth, "Investment growth"),
        (:N, "Employment"),
        (:wgrowth, "Wage growth"),
        (:RB, "Nominal rate"),
        (:π, "Inflation"),
        (:σ, "Income risk"),
        (:Tprog, "Tax progressivity"),
        (:TOP10Wshare, "Top 10 wealth share"),
        (:TOP10Ishare, "Top 10 inc. share"),
    ],
    ShockContr,
    ShockContr_order,
    sr_mc.indexes_r;
    timeline = collect(1954.75:0.25:2019.75),
    show_fig = false,
    save_fig = true,
    path = paths["bld_example"] * "/HDs",
);

mkpath(paths["bld_example"] * "/HDs_cat");
plot_hist_decomp(
    [
        (:Ygrowth, "Output growth"),
        (:Cgrowth, "Consumption growth"),
        (:Igrowth, "Investment growth"),
        (:N, "Employment"),
        (:wgrowth, "Wage growth"),
        (:RB, "Nominal rate"),
        (:π, "Inflation"),
        (:σ, "Income risk"),
        (:Tprog, "Tax progressivity"),
        (:TOP10Wshare, "Top 10 wealth share"),
        (:TOP10Ishare, "Top 10 inc. share"),
    ],
    ShockContr,
    ShockContr_order,
    sr_mc.indexes_r;
    shock_categories = Dict(
        ("Monetary", "mon") => [:Rshock, :A],
        ("Fiscal", "fis") => [:Gshock, :Tprogshock],
        ("Productivity", "pro") => [:Z, :ZI, :μ, :μw],
    ),
    timeline = collect(1954.75:0.25:2019.75),
    show_fig = false,
    save_fig = true,
    path = paths["bld_example"] * "/HDs_cat",
);

@printf "\n"
@printf "Done.\n"


# Save Runtime data

In [ ]:
res_Runtime

Dict{String, Any} with 11 entries:
  "KERNEL"                       => :Darwin
  "t_linearize_full_model"       => 200.898
  "t_call_prepare_linearization" => 570.887
  "MACHINE"                      => "arm64-apple-darwin24.0.0"
  "FREE_MEMORY"                  => "67.062 MiB"
  "JULIA_VERSION"                => v"1.11.9"
  "TOTAL_MEMORY"                 => "24.000 GiB"
  "t_call_find_steadystate"      => 274.148
  "nthreads"                     => 6
  "CPU_NAME"                     => "apple-m2"
  "t_model_reduction"            => 4.48531

old / orignal code

In [ ]:
res_Runtime

Dict{String, Any} with 13 entries:
  "KERNEL"                       => :Darwin
  "t_linearize_full_model"       => 91.1203
  "t_call_prepare_linearization" => 14.2521
  "MACHINE"                      => "arm64-apple-darwin24.0.0"
  "FREE_MEMORY"                  => "73.328 MiB"
  "t_find_mode"                  => (value = (EstimResults([0.705572, 1.94092, …
  "JULIA_VERSION"                => v"1.11.9"
  "TOTAL_MEMORY"                 => "24.000 GiB"
  "t_call_find_steadystate"      => 47.8316
  "nthreads"                     => 6
  "CPU_NAME"                     => "apple-m2"
  "t_model_reduction"            => 4.7776
  "t_sample_posterior"           => 173.212

In [ ]:

mkpath(paths["bld_runtime"]);
date_prefix = Dates.format(Dates.today(), "yyyy-mm-dd");
runtime_file = joinpath(paths["bld_runtime"], date_prefix * "_" * res_Runtime["CPU_NAME" ] * "_" * string(res_Runtime["nthreads"]) * "_" * "res_Runtime.jld2");
# (runtime_file, true; res_Runtime);
println("Saved runtime data to: ", runtime_file);

In [ ]:
res_Runtime

Dict{String, Any} with 7 entries:
  "nthreads"      => 6
  "KERNEL"        => :Darwin
  "CPU_NAME"      => "apple-m2"
  "JULIA_VERSION" => v"1.11.9"
  "MACHINE"       => "arm64-apple-darwin24.0.0"
  "FREE_MEMORY"   => "69.391 MiB"
  "TOTAL_MEMORY"  => "24.000 GiB"